# Use `Lale` `AIF360` scorers to calculate and mitigate bias for credit risk AutoAI model

This notebook contains the steps and code to demonstrate support of AutoAI experiments in watsonx.ai Runtime service. It introduces commands for bias detecting and mitigation performed with `lale.lib.aif360` module.

Some familiarity with Python is helpful. This notebook uses Python 3.12.

## Contents

This notebook contains the following parts:

1. [Set up the environment](#1.-Set-up-the-environment)
2. [Optimizer definition](#2.-Optimizer-definition)
3. [Experiment run](#3.-Experiment-run)
4. [Bias detection and mitigation](#4.-Bias-detection-and-mitigation)
5. [Deploy and score](#5.-Deploy-and-score)
6. [Cleanup](#6.-Cleanup)
7. [Summary and next steps](#7.-Summary-and-next-steps)

<a id="1.-Set-up-the-environment"></a>
## 1. Set up the environment

If you are not familiar with <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> and AutoAI experiments please read more about it in the sample notebook: <a href="https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/experiments/autoai/Use%20AutoAI%20and%20Lale%20to%20predict%20credit%20risk.ipynb" target="_blank" rel="noopener no referrer">"Use AutoAI and Lale to predict credit risk."</a>

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install wget | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1
%pip install "setuptools<81" | tail -n 1 # Needed for `pkg_resources` package
%pip install "snapml>=1.17.2,<1.18.0" | tail -n 1
%pip install "lightgbm>=4.5.0,<4.6.0" | tail -n 1
%pip install "autoai-libs>=3.0.12,<4.0.0" | tail -n 1
%pip install "lale[fairness]>=0.9.2,<0.10.0" | tail -n 1

### Connection to watsonx.ai Runtime

Authenticate the watsonx.ai Runtime service on IBM Cloud. You need to provide platform `api_key` and instance `location`.

You can use [IBM Cloud CLI](https://cloud.ibm.com/docs/cli/index.html) to retrieve platform API Key and instance location.

API Key can be generated in the following way:
```
ibmcloud login
ibmcloud iam api-key-create API_KEY_NAME
```

In result, get the value of `api_key` from the output.


Location of your watsonx.ai Runtime instance can be retrieved in the following way:
```
ibmcloud login --apikey API_KEY -a https://cloud.ibm.com
ibmcloud resource service-instance INSTANCE_NAME
```

In result, get the value of `location` from the output.

**Tip**: Your `Cloud API key` can be generated by going to the [**Users** section of the Cloud console](https://cloud.ibm.com/iam#/users). From that page, click your name, scroll down to the **API Keys** section, and click **Create an IBM Cloud API key**. Give your key a name and click **Create**, then copy the created key and paste it below. You can also get a service specific url by going to the [**Endpoint URLs** section of the watsonx.ai Runtime docs](https://cloud.ibm.com/apidocs/machine-learning).  You can check your instance location in your  <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance details.

You can also get service specific apikey by going to the [**Service IDs** section of the Cloud Console](https://cloud.ibm.com/iam/serviceids).  From that page, click **Create**, then copy the created key and paste it below.

**Action**: Enter your `url` and `api_key` in the following cell.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

In [3]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

You need to create a space that will be used for your work. If you do not have a space, you can use [Deployment Spaces Dashboard](https://dataplatform.cloud.ibm.com/ml-runtime/spaces?context=cpdaas) to create one.

- Click **New Deployment Space**
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press **Create**
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: assign space ID below


In [4]:
space_id = "PASTE YOUR SPACE ID HERE"

You can use the `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in watsonx.ai Runtime, you need to set the **space** which you will be using.

In [5]:
client.set.default_space(space_id)

'SUCCESS'

<a id="2.-Optimizer-definition"></a>
## 2. Optimizer definition

### Training data connection

Define connection information to COS bucket and training data CSV file. This example uses the [German Credit Risk dataset](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/data/credit_risk/credit_risk_training_light.csv).

The code in next cell uploads training data to the bucket.

Download training data from git repository and split for training and test set.

In [6]:
import os

import pandas as pd
import wget

filename = "german_credit_data_biased_training.csv"
url = "https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/data/credit_risk/german_credit_data_biased_training.csv"

if not os.path.isfile(filename):
    wget.download(url)

credit_risk_df = pd.read_csv(filename)
credit_risk_df.head()

,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,OthersOnLoan,...,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker,Risk
0,0_to_200,31,credits_paid_to_date,other,1889,100_to_500,less_1,3,female,none,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
1,less_0,18,credits_paid_to_date,car_new,462,less_100,1_to_4,2,female,none,...,savings_insurance,37,stores,own,2,skilled,1,none,yes,No Risk
2,less_0,15,prior_payments_delayed,furniture,250,less_100,1_to_4,2,male,none,...,real_estate,28,none,own,2,skilled,1,yes,no,No Risk
3,0_to_200,28,credits_paid_to_date,retraining,3693,less_100,greater_7,3,male,none,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
4,no_checking,28,prior_payments_delayed,education,6235,500_to_1000,greater_7,3,male,none,...,unknown,57,none,own,2,skilled,1,none,yes,Risk


Define connection information to training data.

In [7]:
cos_credentials = client.spaces.get_details(space_id=space_id)["entity"]["storage"][
    "properties"
]
datasource_name = "bluemixcloudobjectstorage"
bucket_name = cos_credentials["bucket_name"]

conn_meta_props = {
    client.connections.ConfigurationMetaNames.NAME: f"Connection to Database - {datasource_name} ",
    client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: client.connections.get_datasource_type_id_by_name(
        datasource_name
    ),
    client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection to external Database",
    client.connections.ConfigurationMetaNames.PROPERTIES: {
        "bucket": bucket_name,
        "access_key": cos_credentials["credentials"]["editor"]["access_key_id"],
        "secret_key": cos_credentials["credentials"]["editor"]["secret_access_key"],
        "iam_url": "https://iam.cloud.ibm.com/identity/token",
        "url": cos_credentials["endpoint_url"],
    },
}

conn_details = client.connections.create(meta_props=conn_meta_props)

Creating connections...
SUCCESS


**Note**: The above connection can be initialized alternatively with `api_key` and `resource_instance_id`.  
The above cell can be replaced with:


```python
conn_meta_props= {
    client.connections.ConfigurationMetaNames.NAME: f"Connection to Database - {db_name} ",
    client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: client.connections.get_datasource_type_id_by_name(db_name),
    client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection to external Database",
    client.connections.ConfigurationMetaNames.PROPERTIES: {
        'bucket': bucket_name,
        'api_key': cos_credentials['apikey'],
        'resource_instance_id': cos_credentials['resource_instance_id'],
        'iam_url': 'https://iam.cloud.ibm.com/identity/token',
        'url': 'https://s3.us.cloud-object-storage.appdomain.cloud'
    }
}

conn_details = client.connections.create(meta_props=conn_meta_props)
```

In [8]:
from sklearn.model_selection import train_test_split

X = credit_risk_df.drop(["Risk"], axis=1)
y = credit_risk_df["Risk"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1)

Define connection information to training data and upload train dataset to COS bucket.


In [9]:
from ibm_watsonx_ai.helpers import DataConnection, S3Location

connection_id = client.connections.get_id(conn_details)

credit_risk_conn = DataConnection(
    connection_asset_id=connection_id,
    location=S3Location(bucket=bucket_name, path=filename),
)
credit_risk_conn.set_client(client)
credit_risk_conn.write(
    data=X_train.join(y_train), remote_name=filename, use_flight=False
)

training_data_reference = [credit_risk_conn]

### Optimizer configuration

Provide the input information for AutoAI optimizer:
- `name` - experiment name
- `prediction_type` - type of the problem
- `prediction_column` - target column name
- `scoring` - optimization metric
- `daub_include_only_estimators` - estimators which will be included during AutoAI training. More available estimators can be found in `experiment.ClassificationAlgorithms` enum

In [10]:
from ibm_watsonx_ai.experiment import AutoAI

experiment = AutoAI(credentials, space_id=space_id)

pipeline_optimizer = experiment.optimizer(
    name="Credit Risk Bias detection in AutoAI",
    prediction_type=AutoAI.PredictionType.BINARY,
    prediction_column="Risk",
    scoring=AutoAI.Metrics.ROC_AUC_SCORE,
    include_only_estimators=[experiment.ClassificationAlgorithms.XGB],
)

<a id="3.-Experiment-run"></a>
## 3. Experiment run

Call the `fit()` method to trigger the AutoAI experiment. You can either use interactive mode (synchronous job) or background mode (asynchronous job) by specifying `background_mode=True`.

In [11]:
run_details = pipeline_optimizer.fit(training_data_reference=training_data_reference)

Training job 9722e854-cf48-4997-a302-71b0b85248b6 completed: 100%|████████| [03:56<00:00,  2.37s/it]


You can use the `get_run_status()` method to monitor AutoAI jobs in background mode.

In [12]:
pipeline_optimizer.get_run_status()

'completed'

In [13]:
summary = pipeline_optimizer.summary()
summary

,Enhancements,Estimator,training_roc_auc_(optimized),holdout_average_precision,holdout_log_loss,training_accuracy,holdout_roc_auc,training_balanced_accuracy,training_f1,holdout_precision,training_average_precision,training_log_loss,holdout_recall,training_precision,holdout_accuracy,holdout_balanced_accuracy,training_recall,holdout_f1
Pipeline Name,,,,,,,,,,,,,,,,,,
Pipeline_1,,XGBClassifier,0.839298,0.458675,0.316812,0.788954,0.845260,0.739650,0.848474,0.850746,0.907720,0.456092,0.956376,0.811155,0.859688,0.812625,0.889635,0.900474
Pipeline_2,HPO,XGBClassifier,0.847332,0.483774,0.456055,0.788458,0.832359,0.742194,0.847202,0.812883,0.913619,0.446747,0.889262,0.814340,0.790646,0.742644,0.882923,0.849359
Pipeline_3,"HPO, FE",XGBClassifier,0.842413,0.489655,0.473417,0.779291,0.811381,0.732919,0.840319,0.797546,0.912516,0.453474,0.872483,0.809259,0.768374,0.717699,0.873975,0.833333
Pipeline_4,"HPO, FE, HPO",XGBClassifier,0.842813,0.490273,0.469214,0.779787,0.808803,0.726174,0.842917,0.774481,0.912684,0.452636,0.875839,0.801249,0.748330,0.686264,0.889262,0.822047
Pipeline_5,"HPO, FE, HPO, Ensemble",BatchedTreeEnsembleClassifier(XGBClassifier),0.842813,0.490273,0.469214,0.779787,0.808803,0.726174,0.842917,0.774481,0.912684,0.452636,0.875839,0.801249,0.748330,0.686264,0.889262,0.822047


### Get selected pipeline model

Download pipeline model object from the AutoAI training job.

In [14]:
best_pipeline = pipeline_optimizer.get_pipeline()

  Using cached pyarrow-23.0.0-cp312-cp312-macosx_12_0_x86_64.whl.metadata (3.0 kB)
Using cached pyarrow-23.0.0-cp312-cp312-macosx_12_0_x86_64.whl (35.8 MB)


<a id="4.-Bias-detection-and-mitigation"></a>
## 4. Bias detection and mitigation

The `fairness_info` dictionary contains some fairness-related metadata. The favorable and unfavorable label are values of the target class column that indicate whether the loan was granted or denied. A protected attribute is a feature that partitions the population into groups whose outcome should have parity. The credit-risk dataset has two protected attribute columns, sex and age. Each prottected attributes has monitored and reference group.


In [15]:
fairness_info = {
    "favorable_labels": ["No Risk"],
    "protected_attributes": [
        {
            "feature": "Sex",
            "reference_group": ["male"],
            "monitored_group": ["female"],
        },
        {
            "feature": "Age",
            "reference_group": [[26, 40]],
            "monitored_group": [[18.0, 25.0], [41.0, 75.0]],
        },
    ],
}

### Calculate fairness metrics

We will calculate some model metrics. Accuracy describes how accurate is the model according to dataset. 
Disparate impact is defined by comparing outcomes between a privileged group and an unprivileged group, 
so it needs to check the protected attribute to determine group membership for the sample record at hand.
The third calculated metric takes the disparate impact into account along with accuracy. The best value of the score is 1.0.

In [16]:
import sklearn.metrics
from lale.lib.aif360 import accuracy_and_disparate_impact, disparate_impact

accuracy_scorer = sklearn.metrics.make_scorer(sklearn.metrics.accuracy_score)
print(f"accuracy {accuracy_scorer(best_pipeline, X_test, y_test):.1%}")

disparate_impact_scorer = disparate_impact(**fairness_info)
print(f"disparate impact {disparate_impact_scorer(best_pipeline, X_test, y_test):.2f}")

combined_scorer = accuracy_and_disparate_impact(**fairness_info)
print(
    f"accuracy and disparate impact metric {combined_scorer(best_pipeline, X_test, y_test):.2f}"
)

accuracy 79.6%
disparate impact 0.72
accuracy and disparate impact metric 0.76


### Mitigation

`Hyperopt` minimizes (best_score - score_returned_by_the_scorer), where best_score is an argument to Hyperopt and score_returned_by_the_scorer is the value returned by the scorer for each evaluation point. We will use the `Hyperopt` to tune hyperparametres of the AutoAI pipeline to get new and more fair model. 


In [17]:
from lale import wrap_imported_operators
from lale.lib.aif360 import FairStratifiedKFold
from lale.lib.lale import Hyperopt
from sklearn.linear_model import LogisticRegression as LR
from sklearn.neighbors import KNeighborsClassifier as KNN
from sklearn.tree import DecisionTreeClassifier as Tree

wrap_imported_operators()

In [18]:
prefix = best_pipeline.remove_last().freeze_trainable()
prefix.export_to_sklearn_pipeline()

Pipeline(steps=[('featureunion',
                 FeatureUnion(transformer_list=[('float32_transform_5358101392',
                                                 Pipeline(steps=[('numpycolumnselector',
                                                                  NumpyColumnSelector(columns=[0,
                                                                                               1,
                                                                                               2,
                                                                                               3,
                                                                                               5,
                                                                                               6,
                                                                                               7,
                                                                                               8,
                                                                                               9,
                                                                                               10,
                                                                                               11,
                                                                                               12,
                                                                                               13,
                                                                                               14,
                                                                                               15,
                                                                                               16,
                                                                                               17,
                                                                                               18,
                                                                                               19])),
                                                                 ('compressstrings',
                                                                  CompressStrings(compress_type='hash',
                                                                                  dtypes_list=['char_str',
                                                                                               'int_num',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'i...
                                                                 ('numpyreplacemissingvalues',
                                                                  NumpyReplaceMissingValues(missing_values=[])),
                                                                 ('numimputer',
                                                                  NumImputer(missing_values=nan,
                                                                             strategy='median')),
                                                                 ('optstandardscaler',
                                                                  OptStandardScaler(use_scaler_flag=False)),
                                                                 ('float32_transform',
                                                                  float32_transform())]))])),
                ('numpypermutearray',
                 NumpyPermuteArray(axis=0,
                                   permutation_ind

In [19]:
new_pipeline = prefix >> (LR | Tree | KNN)

In [20]:
fair_cv = FairStratifiedKFold(**fairness_info, n_splits=3)

pipeline_fairer = new_pipeline.auto_configure(
    X_train,
    y_train,
    optimizer=Hyperopt,
    cv=fair_cv,
    max_evals=10,
    scoring=combined_scorer,
    best_score=1.0,
)

100%|██████████| 10/10 [00:07<00:00,  1.30trial/s, best loss: 0.16733333333333322]
1 out of 10 trials failed, call summary() for details.
Run with verbose=True to see per-trial exceptions.


As with any trained model, we can evaluate and visualize the result.

In [21]:
print(f"accuracy {accuracy_scorer(pipeline_fairer, X_test, y_test):.1%}")
print(
    f"disparate impact {disparate_impact_scorer(pipeline_fairer, X_test, y_test):.2f}"
)
print(
    f"accuracy and disparate impact metric {combined_scorer(pipeline_fairer, X_test, y_test):.2f}"
)
pipeline_fairer.export_to_sklearn_pipeline()

accuracy 67.2%
disparate impact 1.00
accuracy and disparate impact metric 0.84


Pipeline(steps=[('featureunion',
                 FeatureUnion(transformer_list=[('float32_transform_5362122976',
                                                 Pipeline(steps=[('numpycolumnselector',
                                                                  NumpyColumnSelector(columns=[0,
                                                                                               1,
                                                                                               2,
                                                                                               3,
                                                                                               5,
                                                                                               6,
                                                                                               7,
                                                                                               8,
                                                                                               9,
                                                                                               10,
                                                                                               11,
                                                                                               12,
                                                                                               13,
                                                                                               14,
                                                                                               15,
                                                                                               16,
                                                                                               17,
                                                                                               18,
                                                                                               19])),
                                                                 ('compressstrings',
                                                                  CompressStrings(compress_type='hash',
                                                                                  dtypes_list=['char_str',
                                                                                               'int_num',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'i...
                                                                  OptStandardScaler(use_scaler_flag=False)),
                                                                 ('float32_transform',
                                                                  float32_transform())]))])),
                ('numpypermutearray',
                 NumpyPermuteArray(axis=0,
                                   permutation_indices=[0, 1, 2, 3, 5, 6, 7, 8,
                                                        9, 10, 11, 12, 13, 14,
                                                        15, 16, 17, 18, 19,
                                                        4])),
                ('decisiontreeclassifier',
                 DecisionTreeClassifier(criterion='entropy',
                                        max_features=0.08317737987226397,
                                        min_samples_leaf=0.35223000813752336,
                                        min_samples_split=0.3003021

As the result demonstrates, the best model found by AI Automation
has lower accuracy and much better disparate impact as the one we saw
before. Also, it has tuned the repair level and
has picked and tuned a classifier. These results may vary by dataset and search space.

You can get source code of the created pipeline by running the snippet below:
```python
pipeline_fairer.pretty_print(ipython_display=True, show_imports=False)
```

<a id="5.-Deploy-and-score"></a>
## 5. Deploy and score
In this section you will learn how to deploy and score Lale pipeline model using watsonx.ai Runtime instance.

### Store the model

In [22]:
model_props = {
    client.repository.ModelMetaNames.NAME: "Fairer AutoAI model",
}
feature_vector = list(X.columns)

Get training's id from run details.

In [23]:
training_id = run_details["metadata"]["id"]

In [24]:
published_model = client.repository.store_model(
    model=best_pipeline.export_to_sklearn_pipeline(),
    meta_props=model_props,
    training_id=training_id,
)

In [25]:
published_model_id = client.repository.get_model_id(published_model)

### Deployment creation

In [26]:
metadata = {
    client.deployments.ConfigurationMetaNames.NAME: "Deployment of fairer model",
    client.deployments.ConfigurationMetaNames.ONLINE: {},
}

created_deployment = client.deployments.create(published_model_id, meta_props=metadata)



######################################################################################

Synchronous deployment creation for id: '0a6b4a43-737c-40c2-a5d0-1a6c88b5125f' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
.....
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='39e7b22a-24ac-4a16-8409-76a5c93fb484'
-----------------------------------------------------------------------------------------------




In [27]:
deployment_id = client.deployments.get_id(created_deployment)

#### Deployment scoring 

You need to pass scoring values as input data if the deployed model. Use `client.deployments.score()` method to get predictions from deployed model. 

In [28]:
values = X_test.values

scoring_payload = {"input_data": [{"values": values[:5]}]}

In [29]:
predictions = client.deployments.score(deployment_id, scoring_payload)
predictions

{'predictions': [{'fields': ['prediction', 'probability'],
   'values': [['No Risk', [0.9925997257232666, 0.007400278467684984]],
    ['No Risk', [0.9002817869186401, 0.09971818327903748]],
    ['Risk', [0.15350157022476196, 0.846498429775238]],
    ['No Risk', [0.654547929763794, 0.34545207023620605]],
    ['Risk', [0.07706844806671143, 0.9229315519332886]]]}]}

<a id="6.-Cleanup"></a>
## 6. Cleanup

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="7.-Summary-and-next-steps"></a>
## 7. Summary and next steps

 You successfully completed this notebook!

Check out used packeges domuntations:
- `ibm-watsonx-ai` [Online Documentation](https://www.ibm.com/cloud/watson-studio/autoai)
- `lale`: https://github.com/IBM/lale
- `aif360`: https://aif360.mybluemix.net/

### Authors 

**Dorota Lączek**, Software Engineer at watsonx.ai

**Mateusz Szewczyk**, Software Engineer at watsonx.ai

Copyright © 2020-2026 IBM. This notebook and its source code are released under the terms of the MIT License.